In [29]:
import sys
sys.path.append('../')
import pandas as pd
import plotly.graph_objects as go
import datetime as dt
from plotting import CandlePlot

In [30]:
from infrastructure.instrument_collection import instrumentCollection as ic

In [31]:
pair = 'EUR_JPY'
granularity = 'H4'
df = pd.read_pickle(f'../data/{pair}_{granularity}.pkl')
MA_LIST = [10,20,50,100,200]

In [32]:
df.columns

Index(['time', 'volume', 'mid_o', 'mid_h', 'mid_l', 'mid_c', 'bid_o', 'bid_h',
       'bid_l', 'bid_c', 'ask_o', 'ask_h', 'ask_l', 'ask_c'],
      dtype='object')

In [33]:
df_ma = df[['time','mid_o','mid_h','mid_l','mid_c']].copy()

In [34]:
df_ma.head()

,time,mid_o,mid_h,mid_l,mid_c
0,2023-01-05 22:00:00+00:00,140.395,140.844,140.230,140.844
1,2023-01-06 02:00:00+00:00,140.846,141.270,140.642,141.078
2,2023-01-06 06:00:00+00:00,141.082,141.449,140.632,141.246
3,2023-01-06 10:00:00+00:00,141.246,141.442,140.430,140.620
4,2023-01-06 14:00:00+00:00,140.615,140.934,140.300,140.546


In [35]:
for ma in MA_LIST:
    df_ma[f'MA_{ma}']=df_ma.mid_c.rolling(window=ma).mean()
df_ma.dropna(inplace=True)
df_ma.reset_index(inplace=True,drop=True)

In [36]:
df_plot=df_ma.iloc[:500]

In [37]:
cp = CandlePlot(df=df_plot)

In [38]:
# cp.fig.add_trace(go.Scatter(
#     x=cp.df_plot.sTime,
#     y=cp.df_plot.MA_10,
#     line=dict(width=2),
#     line_shape='spline',
#     name='MA_10'
# ))
cp.show_plot(line_traces=[ f"MA_{x}"  for x in MA_LIST])

In [39]:
MA_S='MA_10'
MA_L='MA_20'
BUY=1
SELL=-1
NONE=0

In [40]:
df_an = df_ma[['time','mid_o','mid_h','mid_l','mid_c',MA_S,MA_L]].copy()

In [41]:
df_an['DELTA'] = df_an[MA_S]-df_an[MA_L]
df_an['DELTA_PREV']=df_an['DELTA'].shift(1)

In [42]:
df_an.head()

,time,mid_o,mid_h,mid_l,mid_c,MA_10,MA_20,DELTA,DELTA_PREV
0,2023-02-22 02:00:00+00:00,143.588,143.860,143.532,143.662,143.5601,143.46680,0.09330,NaN
1,2023-02-22 06:00:00+00:00,143.660,143.786,143.383,143.514,143.5677,143.47070,0.09700,0.0933
2,2023-02-22 10:00:00+00:00,143.514,143.583,143.062,143.272,143.5475,143.46690,0.08060,0.0970
3,2023-02-22 14:00:00+00:00,143.278,143.430,143.050,143.228,143.5337,143.45730,0.07640,0.0806
4,2023-02-22 18:00:00+00:00,143.226,143.296,143.046,143.094,143.5060,143.44825,0.05775,0.0764


In [43]:
def is_trade(row):
    if row.DELTA>=0 and row.DELTA_PREV<0:
        return BUY
    elif row.DELTA<0 and row.DELTA_PREV>=0:
        return SELL
    return NONE

In [44]:
df_an['TRADE']=df_an.apply(is_trade,axis=1)

In [45]:
df_trades=df_an[df_an.TRADE!=NONE]

In [46]:
df_trades.head()

,time,mid_o,mid_h,mid_l,mid_c,MA_10,MA_20,DELTA,DELTA_PREV,TRADE
6,2023-02-23 02:00:00+00:00,143.152,143.235,143.088,143.230,143.4244,143.42505,-0.00065,0.03205,-1
19,2023-02-27 06:00:00+00:00,143.704,143.972,143.568,143.910,143.3923,143.30515,0.08715,-0.01805,1
47,2023-03-05 22:00:00+00:00,144.526,144.608,144.154,144.416,144.7132,144.76450,-0.05130,0.02530,-1
55,2023-03-07 06:00:00+00:00,145.309,145.310,144.602,144.785,144.8774,144.87195,0.00545,-0.03270,1
64,2023-03-08 18:00:00+00:00,144.683,144.910,144.614,144.820,144.7932,144.82675,-0.03355,0.03770,-1


In [47]:
cp = CandlePlot(df_an.iloc[20:80])
cp.show_plot(line_traces=[MA_S,MA_L])

In [48]:
ic.LoadInstruments('../data')

In [49]:
ic.instruments_dict[pair]

{'name': 'EUR_JPY', 'ins_type': 'CURRENCY', 'diplayName': 'EUR/JPY', 'pipLocation': 0.01, 'tradeUnitsPrecision': 0, 'marginRate': 0.03333333333333}

In [50]:
ins_data = ic.instruments_dict[pair]

In [51]:
df_trades['DIFF'] = df_trades.mid_c.diff().shift(-1)
df_trades.fillna(0,inplace=True)

C:\Users\Teo\AppData\Local\Temp\ipykernel_15408\3447731209.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\Teo\AppData\Local\Temp\ipykernel_15408\3447731209.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [52]:
df_trades['GAIN']=df_trades['DIFF']/ins_data.pipLocation
df_trades['GAIN']=df_trades['GAIN']*df_trades['TRADE']

C:\Users\Teo\AppData\Local\Temp\ipykernel_15408\2606551256.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\Teo\AppData\Local\Temp\ipykernel_15408\2606551256.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [53]:
df_trades.GAIN.sum()

np.float64(-971.0000000000206)

In [54]:
df_trades['GAIN_C'] = df_trades['GAIN'].cumsum()

C:\Users\Teo\AppData\Local\Temp\ipykernel_15408\735232793.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [55]:
cp = CandlePlot(df_trades,candles=False)
cp.show_plot(line_traces=['GAIN_C'])